In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json" # file unico (42 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=8, top_k=20, top_n=7,       # stesso numero di chunk (top_j) per TUTTE le configurazioni
    # gold_path=QDIR / "gold_chunks.json",   # opzionale: {query_id: [chunk_id,...]} -> metriche retrieval senza LLM
)
print("!>> Handler e benchmark pronti")

<<! File sidecar ripristinato allo stato iniziale !>>


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
13:19:59 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
13:19:59 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> Handler e benchmark pronti


In [2]:
### 1 ### Unione dei due file in eval_queries.json (serve a handler.update_query(new_id=...)) + controlli
import json
from pathlib import Path
QDIR = Path("~/tesi_graphrag/data/queries/dst").expanduser()

pred = json.load(open(QDIR / "queries_predetermined.json", encoding="utf-8"))
norm = json.load(open(QDIR / "queries_normal.json", encoding="utf-8"))
allq = pred + norm

ids = [q["id"] for q in allq]
assert len(ids) == len(set(ids)) == 42, "id duplicati o numero query inatteso"
assert all(q["query"].strip() and q["reference"].strip() for q in allq)

json.dump(allq, open(QDIR / "eval_queries.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
q_by_id = {q["id"]: q for q in allq}                 # accesso per chiave, es. q_by_id["CPX_P01"]["reference"]
print(len(pred), "predeterminate +", len(norm), "normali ->", len(allq), "in eval_queries.json")

21 predeterminate + 21 normali -> 42 in eval_queries.json


In [3]:
### 2 ### Registrazione delle predeterminate nel sidecar (ids: predq_DEF_P01, predq_CPX_P01, ...)
registered = bench.register_predetermined()
print(len(registered), "predeterminate registrate (eseguire UNA sola volta per sidecar)")

13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_DEF_P01] |>>
13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_DEF_P02] |>>
13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_DEF_P03] |>>
13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_DEF_P04] |>>
13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_VIN_P01] |>>
13:19:59 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_VIN_P02] |>>
13:20:00 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_VIN_P03] |>>
13:20:00 [INFO] src.predetermined_query_handling_methods: <<| Query Pre-Determinata registrata: ID=[predq_ALG_P01] |>>
13:20:00 [INFO] src.predetermined_query_handling

21 predeterminate registrate (eseguire UNA sola volta per sidecar)


In [4]:
### 3 ### Controllo similarita': nessuna normale deve avvicinarsi a una predeterminata (soglia handler 0.85)
sim = bench.check_similarity(warn_above=0.80)
display(sim.head(10))       # le piu' vicine; quelle >= 0.80 vanno riformulate

13:20:04 [WARNING] src.evaluation.benchmark_runner: !> 1 query normali troppo simili a una predeterminata: ['ALG_N02']


,normal_id,paired_with,max_sim,closest_pred,sim_with_partner
7,ALG_N02,ALG_P02,0.812466,ALG_P02,0.812466
17,CMP_N03,CMP_P01,0.798265,DEF_P01,0.616754
6,ALG_N01,ALG_P03,0.793582,ALG_P02,0.653378
5,VIN_N02,VIN_P01,0.791388,CPX_P04,0.711623
20,CHI_N02,CHI_P01,0.748677,VIN_P02,0.424327
16,CMP_N02,CMP_P03,0.748324,DEF_P01,0.690935
19,CHI_N01,CHI_P01,0.743229,VIN_P02,0.387944
1,DEF_N02,DEF_P02,0.725402,VIN_P01,0.695141
10,ALG_N05,ALG_P05,0.703592,CPX_P04,0.485283
0,DEF_N01,DEF_P01,0.691793,VIN_P01,0.518845
